In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time
import yfinance as yf
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge, ElasticNet
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.offline as pyo
from mpl_toolkits.mplot3d import Axes3D

# XGBoost 시도
try:
    import xgboost as xgb
    HAS_XGB = True
    print("✅ XGBoost 사용 가능")
except ImportError:
    HAS_XGB = False
    print("⚠️ XGBoost 없음")

print("🚀 고성능 삼성전자 예측 시스템 V2.0 통합 버전 로딩!")


In [ ]:
class EnhancedDataCollector:
    """향상된 데이터 수집기"""

    def __init__(self):
        self.market_regime_indicators = {}

    def get_samsung_data_with_volume_profile(self, symbol="005930.KS"):
        """거래량 프로파일이 포함된 삼성전자 데이터"""
        print("📊 향상된 삼성전자 데이터 수집...")

        ticker = yf.Ticker(symbol)
        data = ticker.history(period="max")

        if data.empty:
            raise ValueError("데이터 수집 실패")

        # 인덱스 정규화
        data.index = pd.to_datetime(data.index.date)

        # 향상된 거래량 분석
        data['VWAP'] = (data['Close'] * data['Volume']).cumsum() / data['Volume'].cumsum()
        data['Volume_Price_Trend'] = data['Close'] * data['Volume']
        data['Price_Volume_Ratio'] = data['Close'] / (data['Volume'] + 1)

        # 갭 분석
        data['Gap'] = (data['Open'] - data['Close'].shift(1)) / data['Close'].shift(1)
        data['Gap_Fill_Ratio'] = np.where(
            data['Gap'] > 0,
            (data['Close'] - data['Open']) / (data['High'] - data['Open'] + 0.001),
            (data['Close'] - data['Open']) / (data['Open'] - data['Low'] + 0.001)
        )

        # 캔들스틱 패턴
        data['Body_Size'] = abs(data['Close'] - data['Open']) / data['Open']
        data['Upper_Shadow'] = (data['High'] - np.maximum(data['Open'], data['Close'])) / data['Open']
        data['Lower_Shadow'] = (np.minimum(data['Open'], data['Close']) - data['Low']) / data['Open']
        data['Doji'] = (abs(data['Close'] - data['Open']) / (data['High'] - data['Low'] + 0.001) < 0.1).astype(int)

        print(f"✅ 향상된 삼성전자 데이터: {len(data)}일")
        return data.fillna(method='ffill').fillna(0)

    def get_macro_economic_indicators(self, start_date, end_date):
        """거시경제 지표 수집"""
        print("📈 거시경제 지표 수집...")

        macro_data = {}

        # 중요한 거시경제 지표들
        indicators = {
            'USD_KRW': 'KRW=X',           # 달러-원 환율
            'US_10Y': '^TNX',             # 미국 10년 국채
            'VIX': '^VIX',                # 공포지수
            'DXY': 'DX-Y.NYB',            # 달러인덱스
            'GOLD': 'GC=F',               # 금
            'OIL': 'CL=F',                # 원유
            'KOSPI': '^KS11',             # 코스피
            'NASDAQ': '^IXIC',            # 나스닥
            'SP500': '^GSPC',             # S&P500
            'NIKKEI': '^N225',            # 닛케이
            'COPPER': 'HG=F',             # 구리 (경기선행)
            'SEMICONDUCTOR_ETF': 'SMH'     # 반도체 ETF
        }

        for name, symbol in indicators.items():
            try:
                ticker = yf.Ticker(symbol)
                data = ticker.history(start=start_date, end=end_date)

                if not data.empty:
                    data.index = pd.to_datetime(data.index.date)
                    macro_data[name] = data['Close']

            except Exception as e:
                print(f"⚠️ {name} 수집 실패: {e}")

        print(f"✅ 거시경제 지표: {len(macro_data)}개")
        return macro_data


In [ ]:
class AdvancedTechnicalAnalyzer:
    """고급 기술적 분석기"""

    @staticmethod
    def calculate_market_structure_indicators(data):
        """시장 구조 지표 계산"""
        df = data.copy()

        # 고급 이동평균 시스템
        periods = [5, 10, 20, 50, 100, 200]
        for period in periods:
            if len(df) > period:
                # EMA
                df[f'EMA_{period}'] = df['Close'].ewm(span=period, adjust=False).mean()

                # EMA 기울기 (모멘텀)
                df[f'EMA_{period}_slope'] = df[f'EMA_{period}'].diff() / df[f'EMA_{period}']

                # 가격과 EMA 거리
                df[f'Price_EMA_{period}_dist'] = (df['Close'] - df[f'EMA_{period}']) / df[f'EMA_{period}']

        # 이동평균 배열 점수 (상승/하락 추세 강도)
        ma_score = 0
        for i, period in enumerate(periods[:-1]):
            next_period = periods[i + 1]
            if f'EMA_{period}' in df.columns and f'EMA_{next_period}' in df.columns:
                ma_score += (df[f'EMA_{period}'] > df[f'EMA_{next_period}']).astype(int)

        df['MA_Alignment_Score'] = ma_score / (len(periods) - 1)

        # 고급 RSI 변형
        for period in [14, 21]:
            if len(df) > period:
                delta = df['Close'].diff()
                gain = delta.where(delta > 0, 0).rolling(period, min_periods=1).mean()
                loss = -delta.where(delta < 0, 0).rolling(period, min_periods=1).mean()
                rs = gain / (loss + 1e-10)
                rsi = 100 - (100 / (1 + rs))
                df[f'RSI_{period}'] = rsi

                # RSI 발산 감지
                df[f'RSI_{period}_divergence'] = (
                        (df['Close'].diff() > 0) & (rsi.diff() < 0) |
                        (df['Close'].diff() < 0) & (rsi.diff() > 0)
                ).astype(int)

        # 볼린저 밴드 고급 분석
        if len(df) > 20:
            bb_period = 20
            bb_std = 2

            sma = df['Close'].rolling(bb_period).mean()
            std = df['Close'].rolling(bb_period).std()

            df['BB_Upper'] = sma + (std * bb_std)
            df['BB_Lower'] = sma - (std * bb_std)
            df['BB_Width'] = (df['BB_Upper'] - df['BB_Lower']) / sma
            df['BB_Position'] = (df['Close'] - df['BB_Lower']) / (df['BB_Upper'] - df['BB_Lower'])

            # 볼린저 밴드 압축/확장
            df['BB_Squeeze'] = (df['BB_Width'] < df['BB_Width'].rolling(50).quantile(0.2)).astype(int)
            df['BB_Expansion'] = (df['BB_Width'] > df['BB_Width'].rolling(50).quantile(0.8)).astype(int)

        # 고급 MACD
        if len(df) > 26:
            exp1 = df['Close'].ewm(span=12).mean()
            exp2 = df['Close'].ewm(span=26).mean()
            macd = exp1 - exp2
            signal = macd.ewm(span=9).mean()
            histogram = macd - signal

            df['MACD'] = macd
            df['MACD_Signal'] = signal
            df['MACD_Histogram'] = histogram

            # MACD 발산
            df['MACD_Bullish_Divergence'] = (
                    (df['Close'].diff() < 0) & (macd.diff() > 0)
            ).astype(int)
            df['MACD_Bearish_Divergence'] = (
                    (df['Close'].diff() > 0) & (macd.diff() < 0)
            ).astype(int)

        # 스토캐스틱 오실레이터
        if len(df) > 14:
            low_14 = df['Low'].rolling(14).min()
            high_14 = df['High'].rolling(14).max()
            k_percent = 100 * ((df['Close'] - low_14) / (high_14 - low_14 + 1e-10))
            d_percent = k_percent.rolling(3).mean()

            df['Stoch_K'] = k_percent
            df['Stoch_D'] = d_percent
            df['Stoch_Overbought'] = (k_percent > 80).astype(int)
            df['Stoch_Oversold'] = (k_percent < 20).astype(int)

        return df.fillna(method='ffill').fillna(0)

    @staticmethod
    def calculate_momentum_indicators(data):
        """모멘텀 지표"""
        df = data.copy()

        # 다양한 기간의 모멘텀
        for period in [1, 3, 5, 10, 20, 50]:
            if len(df) > period:
                # 가격 모멘텀
                df[f'Price_Momentum_{period}'] = df['Close'].pct_change(period)

                # 거래량 모멘텀
                df[f'Volume_Momentum_{period}'] = df['Volume'].pct_change(period)

                # 가격-거래량 결합 모멘텀
                df[f'PV_Momentum_{period}'] = (
                        df[f'Price_Momentum_{period}'] * df[f'Volume_Momentum_{period}']
                ).fillna(0)

        # ROC (Rate of Change)
        for period in [10, 20]:
            if len(df) > period:
                df[f'ROC_{period}'] = ((df['Close'] - df['Close'].shift(period)) /
                                       df['Close'].shift(period)) * 100

        # 변동성 조정 모멘텀
        for period in [10, 20]:
            if len(df) > period:
                returns = df['Close'].pct_change()
                volatility = returns.rolling(period).std()
                df[f'Vol_Adj_Momentum_{period}'] = (
                        returns.rolling(period).mean() / (volatility + 1e-10)
                )

        return df.fillna(0)


In [ ]:
class SmartFeatureEngineering:
    """지능형 피처 엔지니어링"""

    @staticmethod
    def detect_market_regimes(data):
        """시장 체제 감지"""
        df = data.copy()

        # 변동성 체제
        returns = df['Close'].pct_change().fillna(0)
        volatility = returns.rolling(20).std()

        # 높은/낮은 변동성 체제
        vol_threshold = volatility.rolling(252).quantile(0.7)  # 1년 기준 70분위
        df['High_Vol_Regime'] = (volatility > vol_threshold).astype(int)

        # 추세 체제 (20일/50일 이평선 기준)
        if 'EMA_20' in df.columns and 'EMA_50' in df.columns:
            df['Bull_Regime'] = (df['EMA_20'] > df['EMA_50']).astype(int)
            df['Bear_Regime'] = (df['EMA_20'] < df['EMA_50']).astype(int)

        # 가격 위치 (52주 고점/저점 대비)
        high_52w = df['High'].rolling(252).max()
        low_52w = df['Low'].rolling(252).min()
        df['Price_Position_52W'] = (df['Close'] - low_52w) / (high_52w - low_52w + 1e-10)

        # 거래량 체제
        volume_ma = df['Volume'].rolling(50).mean()
        df['High_Volume_Regime'] = (df['Volume'] > volume_ma * 1.5).astype(int)

        return df

    @staticmethod
    def create_interaction_features(data):
        """상호작용 피처 생성"""
        df = data.copy()

        # 가격-거래량 상호작용
        if 'Volume' in df.columns:
            # 가격과 거래량의 상관관계 (롤링)
            for period in [10, 20]:
                price_vol_corr = df['Close'].rolling(period).corr(df['Volume'])
                df[f'Price_Volume_Corr_{period}'] = price_vol_corr.fillna(0)

        # RSI와 볼린저 밴드 결합
        if 'RSI_14' in df.columns and 'BB_Position' in df.columns:
            df['RSI_BB_Combined'] = df['RSI_14'] * df['BB_Position']

        # MACD와 가격 모멘텀 결합
        if 'MACD' in df.columns and 'Price_Momentum_10' in df.columns:
            df['MACD_Momentum_Combined'] = df['MACD'] * df['Price_Momentum_10']

        # 이동평균 기울기 결합
        ema_slopes = [col for col in df.columns if 'EMA' in col and 'slope' in col]
        if len(ema_slopes) >= 2:
            df['EMA_Slope_Convergence'] = df[ema_slopes[0]] - df[ema_slopes[-1]]

        return df

    @staticmethod
    def create_regime_dependent_features(data):
        """체제별 적응형 피처"""
        df = data.copy()

        # 체제별 RSI 조정
        if 'RSI_14' in df.columns and 'High_Vol_Regime' in df.columns:
            # 높은 변동성 체제에서는 RSI 임계값 조정
            df['Adaptive_RSI_Signal'] = np.where(
                df['High_Vol_Regime'] == 1,
                np.where(df['RSI_14'] > 85, 1, np.where(df['RSI_14'] < 15, -1, 0)),
                np.where(df['RSI_14'] > 70, 1, np.where(df['RSI_14'] < 30, -1, 0))
            )

        # 체제별 모멘텀 가중치
        if 'Bull_Regime' in df.columns and 'Price_Momentum_10' in df.columns:
            df['Regime_Weighted_Momentum'] = (
                    df['Price_Momentum_10'] * (1 + df['Bull_Regime'] * 0.5)
            )

        return df


In [ ]:
class MultiTimeframeEnsemble:
    """다중 시간프레임 앙상블"""

    def __init__(self):
        self.models = {}
        self.scalers = {}
        self.cv_scores = {}

    def prepare_multi_timeframe_data(self, data, target_days=1):
        """다중 시간프레임 데이터 준비"""
        print("🔄 다중 시간프레임 데이터 준비...")

        # 기본 일일 데이터
        daily_data = data.copy()

        # 주간 데이터 (매주 금요일)
        weekly_data = data.resample('W-FRI').agg({
            'Open': 'first',
            'High': 'max',
            'Low': 'min',
            'Close': 'last',
            'Volume': 'sum'
        }).dropna()

        # 월간 데이터
        monthly_data = data.resample('M').agg({
            'Open': 'first',
            'High': 'max',
            'Low': 'min',
            'Close': 'last',
            'Volume': 'sum'
        }).dropna()

        return {
            'daily': daily_data,
            'weekly': weekly_data,
            'monthly': monthly_data
        }

    def create_advanced_models(self):
        """고급 모델 생성"""
        models = {}

        # XGBoost (가능한 경우)
        if HAS_XGB:
            models['xgb'] = xgb.XGBRegressor(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1
            )

        # Random Forest (최적화)
        models['rf'] = RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features='sqrt',
            random_state=42,
            n_jobs=-1
        )

        # Gradient Boosting (최적화)
        models['gbm'] = GradientBoostingRegressor(
            n_estimators=150,
            learning_rate=0.05,
            max_depth=8,
            min_samples_split=10,
            min_samples_leaf=4,
            subsample=0.8,
            random_state=42
        )

        # ElasticNet (정규화)
        models['elastic'] = ElasticNet(
            alpha=0.1,
            l1_ratio=0.5,
            random_state=42
        )

        return models

    def train_with_cross_validation(self, X, y, feature_names):
        """교차 검증을 통한 모델 훈련"""
        print("🔄 교차 검증 모델 훈련...")

        # 시계열 교차 검증
        tscv = TimeSeriesSplit(n_splits=5)

        models = self.create_advanced_models()
        trained_models = {}
        cv_scores = {}

        # 스케일러
        scaler = RobustScaler()
        X_scaled = scaler.fit_transform(X)
        self.scalers['main'] = scaler

        for name, model in models.items():
            print(f"   📚 {name.upper()} 교차 검증 중...")

            cv_score_list = []
            model_instances = []

            # 각 폴드에서 모델 훈련
            for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
                X_train_fold = X_scaled[train_idx] if name == 'elastic' else X.iloc[train_idx]
                X_val_fold = X_scaled[val_idx] if name == 'elastic' else X.iloc[val_idx]
                y_train_fold = y.iloc[train_idx]
                y_val_fold = y.iloc[val_idx]

                # 모델 복사 및 훈련
                from sklearn.base import clone
                model_fold = clone(model)
                model_fold.fit(X_train_fold, y_train_fold)

                # 검증 점수
                val_pred = model_fold.predict(X_val_fold)
                val_score = r2_score(y_val_fold, val_pred)
                cv_score_list.append(val_score)
                model_instances.append(model_fold)

            # 평균 교차 검증 점수
            cv_scores[name] = np.mean(cv_score_list)

            # 전체 데이터로 최종 모델 훈련
            final_X = X_scaled if name == 'elastic' else X
            model.fit(final_X, y)
            trained_models[name] = model

            print(f"   ✅ {name.upper()} CV Score: {cv_scores[name]:.4f}")

        self.models = trained_models
        self.cv_scores = cv_scores
        return cv_scores

    def predict_ensemble(self, X):
        """앙상블 예측"""
        predictions = {}

        for name, model in self.models.items():
            try:
                if name == 'elastic':
                    X_input = self.scalers['main'].transform(X)
                else:
                    X_input = X

                pred = model.predict(X_input)
                predictions[name] = pred

            except Exception as e:
                print(f"⚠️ {name} 예측 실패: {e}")

        if predictions:
            # 가중 평균 (CV 점수 기반)
            weights = {}
            total_weight = 0
            for name in predictions.keys():
                weight = max(0, self.cv_scores.get(name, 0))
                weights[name] = weight
                total_weight += weight

            if total_weight > 0:
                ensemble_pred = np.zeros_like(list(predictions.values())[0])
                for name, pred in predictions.items():
                    ensemble_pred += pred * (weights[name] / total_weight)
            else:
                ensemble_pred = np.mean(list(predictions.values()), axis=0)

            predictions['ensemble'] = ensemble_pred

        return predictions


In [ ]:
class PredictionVisualizer:
    """예측 결과 시각화"""

    def __init__(self, results):
        self.results = results
        self.data = results['data'] if results else None
        self.predictions = results['predictions'] if results else None
        self.ensemble = results['ensemble'] if results else None

    def create_future_predictions(self, days_ahead=30):
        """미래 예측 생성"""
        if not self.data or not self.ensemble:
            return None

        print(f"🔮 {days_ahead}일 미래 예측 생성 중...")

        # 마지막 데이터 포인트
        last_data = self.data.iloc[-1:].copy()

        # 피처 컬럼 확인
        exclude_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'target', 'VWAP']
        feature_cols = [col for col in self.data.columns if col not in exclude_cols]

        # 미래 예측을 위한 시뮬레이션
        future_predictions = []
        future_dates = []
        future_prices = []

        current_price = last_data['Close'].iloc[0]
        current_date = self.data.index[-1]

        # 최근 변동성 계산
        recent_returns = self.data['Close'].pct_change().dropna()[-30:]
        volatility = recent_returns.std()

        for day in range(1, days_ahead + 1):
            # 미래 날짜 생성 (영업일 기준)
            future_date = current_date + pd.Timedelta(days=day)
            while future_date.weekday() > 4:  # 주말 건너뛰기
                future_date += pd.Timedelta(days=1)

            # 마지막 피처 값 사용 (약간의 노이즈 추가)
            X_future = last_data[feature_cols].copy()

            # 일부 피처에 시간 경과 효과 반영
            for col in feature_cols:
                if 'Momentum' in col or 'RSI' in col or 'MACD' in col:
                    # 모멘텀 지표들은 점진적으로 중립화
                    X_future[col] *= (1 - day * 0.01)
                elif 'Vol' in col and 'Regime' in col:
                    # 변동성 체제는 일정 확률로 변화
                    if np.random.random() < 0.05:  # 5% 확률로 체제 변화
                        X_future[col] = 1 - X_future[col]

            # 예측 수행
            try:
                best_model_name = list(self.predictions.keys())[0]  # 첫 번째 모델 사용

                if best_model_name == 'elastic':
                    X_scaled = self.ensemble.scalers['main'].transform(X_future)
                    pred = self.ensemble.models[best_model_name].predict(X_scaled)[0]
                else:
                    pred = self.ensemble.models[best_model_name].predict(X_future)[0]

                # 예측값에 불확실성 추가 (시간이 지날수록 증가)
                uncertainty = volatility * np.sqrt(day) * np.random.normal(0, 0.5)
                pred_adjusted = pred + uncertainty

                # 극단값 제한
                pred_adjusted = np.clip(pred_adjusted, -0.1, 0.1)

                future_predictions.append(pred_adjusted)
                future_dates.append(future_date)

                # 가격 계산
                current_price *= (1 + pred_adjusted)
                future_prices.append(current_price)

            except Exception as e:
                print(f"⚠️ {day}일 예측 실패: {e}")
                break

        return {
            'dates': future_dates,
            'predictions': future_predictions,
            'prices': future_prices
        }

    def plot_comprehensive_analysis(self, days_ahead=30, save_html=True):
        """종합 분석 차트"""
        if not self.data:
            print("❌ 데이터가 없습니다.")
            return

        print("📊 종합 분석 차트 생성 중...")

        # 미래 예측 생성
        future_data = self.create_future_predictions(days_ahead)

        # 최근 100일 데이터
        recent_data = self.data.tail(100).copy()

        # 서브플롯 생성
        fig = make_subplots(
            rows=4, cols=2,
            subplot_titles=[
                '📈 주가 & 미래 예측', '📊 거래량 & 예측 신호',
                '🎯 예측 수익률 분포', '📈 기술적 지표 (RSI)',
                '🌍 거시경제 영향 (환율)', '🧠 모델 성능 비교',
                '📅 미래 30일 시나리오', '⚡ 실시간 신호'
            ],
            specs=[
                [{"secondary_y": True}, {"secondary_y": True}],
                [{"type": "histogram"}, {"secondary_y": True}],
                [{"secondary_y": True}, {"type": "bar"}],
                [{"secondary_y": True}, {"type": "indicator"}]
            ],
            vertical_spacing=0.08,
            horizontal_spacing=0.1
        )

        # 1. 주가 & 미래 예측
        fig.add_trace(
            go.Scatter(
                x=recent_data.index,
                y=recent_data['Close'],
                name='실제 주가',
                line=dict(color='blue', width=2),
                mode='lines'
            ),
            row=1, col=1
        )

        # 이동평균선들
        if 'EMA_20' in recent_data.columns:
            fig.add_trace(
                go.Scatter(
                    x=recent_data.index,
                    y=recent_data['EMA_20'],
                    name='EMA 20',
                    line=dict(color='orange', width=1, dash='dash'),
                    mode='lines'
                ),
                row=1, col=1
            )

        if 'EMA_50' in recent_data.columns:
            fig.add_trace(
                go.Scatter(
                    x=recent_data.index,
                    y=recent_data['EMA_50'],
                    name='EMA 50',
                    line=dict(color='red', width=1, dash='dash'),
                    mode='lines'
                ),
                row=1, col=1
            )

        # 미래 예측
        if future_data:
            fig.add_trace(
                go.Scatter(
                    x=future_data['dates'],
                    y=future_data['prices'],
                    name='미래 예측',
                    line=dict(color='green', width=3, dash='dot'),
                    mode='lines+markers',
                    marker=dict(size=6)
                ),
                row=1, col=1
            )

            # 예측 불확실성 영역
            upper_bound = [p * 1.1 for p in future_data['prices']]
            lower_bound = [p * 0.9 for p in future_data['prices']]

            fig.add_trace(
                go.Scatter(
                    x=future_data['dates'] + future_data['dates'][::-1],
                    y=upper_bound + lower_bound[::-1],
                    fill='toself',
                    fillcolor='rgba(0,255,0,0.2)',
                    line=dict(color='rgba(255,255,255,0)'),
                    name='예측 불확실성',
                    showlegend=False
                ),
                row=1, col=1
            )

        # 2. 거래량 & 예측 신호
        fig.add_trace(
            go.Bar(
                x=recent_data.index,
                y=recent_data['Volume'],
                name='거래량',
                marker_color='lightblue',
                opacity=0.7
            ),
            row=1, col=2
        )

        # 매수/매도 신호
        if self.predictions:
            test_start_idx = int(len(self.data) * 0.8)
            signal_data = self.data.iloc[test_start_idx:].copy()
            best_model = list(self.predictions.keys())[0]

            if len(self.predictions[best_model]) == len(signal_data):
                buy_signals = signal_data[np.array(self.predictions[best_model]) > 0.01]
                sell_signals = signal_data[np.array(self.predictions[best_model]) < -0.01]

                if not buy_signals.empty:
                    fig.add_trace(
                        go.Scatter(
                            x=buy_signals.index,
                            y=buy_signals['Close'],
                            mode='markers',
                            marker=dict(symbol='triangle-up', size=12, color='green'),
                            name='매수 신호'
                        ),
                        row=1, col=2, secondary_y=True
                    )

                if not sell_signals.empty:
                    fig.add_trace(
                        go.Scatter(
                            x=sell_signals.index,
                            y=sell_signals['Close'],
                            mode='markers',
                            marker=dict(symbol='triangle-down', size=12, color='red'),
                            name='매도 신호'
                        ),
                        row=1, col=2, secondary_y=True
                    )

        # 3. 예측 수익률 분포
        if self.predictions:
            all_predictions = []
            for model_name, preds in self.predictions.items():
                all_predictions.extend(preds)

            fig.add_trace(
                go.Histogram(
                    x=all_predictions,
                    nbinsx=50,
                    name='예측 분포',
                    marker_color='purple',
                    opacity=0.7
                ),
                row=2, col=1
            )

        # 4. 기술적 지표 (RSI)
        if 'RSI_14' in recent_data.columns:
            fig.add_trace(
                go.Scatter(
                    x=recent_data.index,
                    y=recent_data['RSI_14'],
                    name='RSI(14)',
                    line=dict(color='purple', width=2),
                    mode='lines'
                ),
                row=2, col=2
            )

            # RSI 과매수/과매도 라인
            fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=2)
            fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=2)

        # 5. 거시경제 영향 (환율)
        macro_cols = [col for col in recent_data.columns if 'macro_USD_KRW' in col]
        if macro_cols:
            fig.add_trace(
                go.Scatter(
                    x=recent_data.index,
                    y=recent_data[macro_cols[0]],
                    name='USD/KRW',
                    line=dict(color='orange', width=2),
                    mode='lines'
                ),
                row=3, col=1
            )

        # 6. 모델 성능 비교
        if hasattr(self, 'results') and 'results' in self.results:
            model_names = list(self.results['results'].index)
            r2_scores = list(self.results['results']['R2'])

            fig.add_trace(
                go.Bar(
                    x=model_names,
                    y=r2_scores,
                    name='R² Score',
                    marker_color=['gold' if i == 0 else 'lightblue' for i in range(len(model_names))]
                ),
                row=3, col=2
            )

        # 7. 미래 30일 시나리오
        if future_data:
            # 상승/하락 시나리오
            optimistic = [p * 1.15 for p in future_data['prices']]
            pessimistic = [p * 0.85 for p in future_data['prices']]

            fig.add_trace(
                go.Scatter(
                    x=future_data['dates'],
                    y=optimistic,
                    name='낙관적 시나리오',
                    line=dict(color='green', width=2, dash='dash'),
                    mode='lines'
                ),
                row=4, col=1
            )

            fig.add_trace(
                go.Scatter(
                    x=future_data['dates'],
                    y=future_data['prices'],
                    name='기본 예측',
                    line=dict(color='blue', width=3),
                    mode='lines'
                ),
                row=4, col=1
            )

            fig.add_trace(
                go.Scatter(
                    x=future_data['dates'],
                    y=pessimistic,
                    name='비관적 시나리오',
                    line=dict(color='red', width=2, dash='dash'),
                    mode='lines'
                ),
                row=4, col=1
            )

        # 8. 실시간 신호 게이지
        latest_pred = 0
        if self.predictions:
            latest_pred = list(self.predictions.values())[0][-1] * 100

        fig.add_trace(
            go.Indicator(
                mode="gauge+number+delta",
                value=latest_pred,
                domain={'x': [0, 1], 'y': [0, 1]},
                title={'text': "예측 수익률(%)"},
                delta={'reference': 0},
                gauge={
                    'axis': {'range': [-5, 5]},
                    'bar': {'color': "darkblue"},
                    'steps': [
                        {'range': [-5, -1], 'color': "lightgray"},
                        {'range': [-1, 1], 'color': "gray"},
                        {'range': [1, 5], 'color': "lightgreen"}
                    ],
                    'threshold': {
                        'line': {'color': "red", 'width': 4},
                        'thickness': 0.75,
                        'value': latest_pred
                    }
                }
            ),
            row=4, col=2
        )

        # 레이아웃 업데이트
        fig.update_layout(
            title=dict(
                text="🚀 삼성전자 AI 예측 시스템 - 종합 분석 대시보드",
                x=0.5,
                font=dict(size=20, color='darkblue')
            ),
            height=1200,
            showlegend=True,
            template="plotly_white",
            font=dict(size=10)
        )

        # Y축 레이블
        fig.update_yaxes(title_text="주가 (원)", row=1, col=1)
        fig.update_yaxes(title_text="거래량", row=1, col=2)
        fig.update_yaxes(title_text="빈도", row=2, col=1)
        fig.update_yaxes(title_text="RSI", row=2, col=2)
        fig.update_yaxes(title_text="환율", row=3, col=1)
        fig.update_yaxes(title_text="R² Score", row=3, col=2)
        fig.update_yaxes(title_text="예상 주가", row=4, col=1)

        # HTML 저장
        if save_html:
            fig.write_html("samsung_prediction_dashboard.html")
            print("💾 대시보드가 'samsung_prediction_dashboard.html'로 저장되었습니다.")

        fig.show()
        return fig

    def plot_3d_prediction_surface(self):
        """3D 예측 표면 차트"""
        if not self.data or not self.predictions:
            print("❌ 데이터나 예측이 없습니다.")
            return

        print("🌐 3D 예측 표면 생성 중...")

        # 테스트 데이터 인덱스
        test_start_idx = int(len(self.data) * 0.8)
        test_data = self.data.iloc[test_start_idx:].copy()

        if len(test_data) < 10:
            print("❌ 테스트 데이터가 부족합니다.")
            return

        # 3D 데이터 준비
        dates = test_data.index
        prices = test_data['Close'].values

        # 주요 모델의 예측값
        main_model = list(self.predictions.keys())[0]
        predictions = self.predictions[main_model]

        # RSI 값 (Z축)
        rsi_col = 'RSI_14' if 'RSI_14' in test_data.columns else None
        rsi_values = test_data[rsi_col].values if rsi_col else np.random.uniform(30, 70, len(test_data))

        # 3D 산점도 생성
        fig = go.Figure()

        # 실제 데이터 포인트
        fig.add_trace(
            go.Scatter3d(
                x=list(range(len(dates))),
                y=prices,
                z=rsi_values,
                mode='markers+lines',
                marker=dict(
                    size=8,
                    color=predictions,
                    colorscale='RdYlGn',
                    colorbar=dict(title="예측 수익률"),
                    showscale=True
                ),
                line=dict(color='blue', width=4),
                name='실제 데이터',
                text=[f"날짜: {d.strftime('%Y-%m-%d')}<br>가격: {p:,.0f}원<br>RSI: {r:.1f}<br>예측: {pred:.3f}"
                      for d, p, r, pred in zip(dates, prices, rsi_values, predictions)],
                hovertemplate="%{text}<extra></extra>"
            )
        )

        # 예측 표면 생성
        x_surf = np.arange(len(dates))
        y_surf = np.linspace(prices.min(), prices.max(), 20)
        X_surf, Y_surf = np.meshgrid(x_surf, y_surf)

        # RSI에 따른 예측 표면 (시뮬레이션)
        Z_surf = np.zeros_like(X_surf)
        for i in range(len(x_surf)):
            for j in range(len(y_surf)):
                # 가격과 RSI 조합에 따른 예측값 시뮬레이션
                price_factor = (Y_surf[j, i] - prices.mean()) / prices.std()
                time_factor = i / len(dates)
                Z_surf[j, i] = 30 + 40 * np.sin(price_factor) + 10 * time_factor

        # 예측 표면 추가
        fig.add_trace(
            go.Surface(
                x=X_surf,
                y=Y_surf,
                z=Z_surf,
                opacity=0.3,
                colorscale='Viridis',
                showscale=False,
                name='예측 표면'
            )
        )

        # 레이아웃
        fig.update_layout(
            title=dict(
                text="🌐 삼성전자 3D 예측 분석<br><sub>시간 × 주가 × RSI 관계</sub>",
                x=0.5,
                font=dict(size=16)
            ),
            scene=dict(
                xaxis_title="시간 순서",
                yaxis_title="주가 (원)",
                zaxis_title="RSI",
                camera=dict(
                    eye=dict(x=1.2, y=1.2, z=1.2)
                )
            ),
            width=900,
            height=700
        )

        fig.show()
        return fig

    def plot_prediction_timeline(self, days_ahead=30):
        """예측 타임라인 차트"""
        if not self.data:
            return

        print("📅 예측 타임라인 생성 중...")

        # 미래 예측
        future_data = self.create_future_predictions(days_ahead)

        # 최근 30일 + 미래 30일
        recent_data = self.data.tail(30)

        fig = go.Figure()

        # 과거 데이터
        fig.add_trace(
            go.Scatter(
                x=recent_data.index,
                y=recent_data['Close'],
                mode='lines+markers',
                name='과거 주가',
                line=dict(color='blue', width=3),
                marker=dict(size=6)
            )
        )

        # 미래 예측
        if future_data:
            fig.add_trace(
                go.Scatter(
                    x=future_data['dates'],
                    y=future_data['prices'],
                    mode='lines+markers',
                    name='미래 예측',
                    line=dict(color='red', width=3, dash='dash'),
                    marker=dict(size=8, symbol='diamond')
                )
            )

            # 현재 시점 표시
            current_price = recent_data['Close'].iloc[-1]
            fig.add_vline(
                x=recent_data.index[-1],
                line_dash="dash",
                line_color="green",
                annotation_text="현재",
                annotation_position="top"
            )

            # 예측 시작점 연결
            fig.add_trace(
                go.Scatter(
                    x=[recent_data.index[-1], future_data['dates'][0]],
                    y=[current_price, future_data['prices'][0]],
                    mode='lines',
                    line=dict(color='orange', width=2, dash='dot'),
                    showlegend=False
                )
            )

        fig.update_layout(
            title="📈 삼성전자 주가 예측 타임라인",
            xaxis_title="날짜",
            yaxis_title="주가 (원)",
            template="plotly_white",
            height=500
        )

        fig.show()
        return fig


In [ ]:
def run_enhanced_system():
    """향상된 시스템 실행"""
    print("🚀 고성능 삼성전자 예측 시스템 V2.0 시작!")
    print("=" * 60)

    try:
        # 1. 데이터 수집
        print("\n📊 1단계: 향상된 데이터 수집")
        collector = EnhancedDataCollector()

        # 삼성전자 데이터
        samsung_data = collector.get_samsung_data_with_volume_profile()
        start_date = samsung_data.index.min()
        end_date = samsung_data.index.max()

        # 거시경제 지표
        macro_data = collector.get_macro_economic_indicators(start_date, end_date)

        # 2. 고급 기술적 분석
        print("\n📈 2단계: 고급 기술적 분석")
        tech_analyzer = AdvancedTechnicalAnalyzer()

        samsung_with_tech = tech_analyzer.calculate_market_structure_indicators(samsung_data)
        samsung_with_momentum = tech_analyzer.calculate_momentum_indicators(samsung_with_tech)

        # 3. 지능형 피처 엔지니어링
        print("\n🧬 3단계: 지능형 피처 엔지니어링")
        feature_eng = SmartFeatureEngineering()

        # 거시 데이터 통합
        for name, series in macro_data.items():
            try:
                series_resampled = series.reindex(samsung_with_momentum.index, method='ffill')
                samsung_with_momentum[f'macro_{name}'] = series_resampled
            except:
                pass

        # 피처 엔지니어링
        samsung_with_regimes = feature_eng.detect_market_regimes(samsung_with_momentum)
        samsung_with_interactions = feature_eng.create_interaction_features(samsung_with_regimes)
        final_data = feature_eng.create_regime_dependent_features(samsung_with_interactions)

        # 데이터 정리
        final_data = final_data.fillna(method='ffill').fillna(method='bfill').fillna(0)
        final_data = final_data.replace([np.inf, -np.inf], 0)

        print(f"✅ 최종 데이터: {final_data.shape}")

        # 4. 타겟 생성 및 데이터 준비
        print("\n🎯 4단계: 타겟 생성 및 데이터 준비")

        # 다음날 수익률 예측
        target_days = 1
        final_data['target'] = final_data['Close'].shift(-target_days).pct_change(target_days)
        final_data = final_data.dropna()

        if len(final_data) < 100:
            raise ValueError("충분한 데이터가 없습니다.")

        # 피처 선택
        exclude_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'target', 'VWAP']
        feature_cols = [col for col in final_data.columns if col not in exclude_cols]

        # 데이터 분할 (최근 20% 테스트)
        split_idx = int(len(final_data) * 0.8)
        train_data = final_data.iloc[:split_idx]
        test_data = final_data.iloc[split_idx:]

        X_train = train_data[feature_cols]
        y_train = train_data['target']
        X_test = test_data[feature_cols]
        y_test = test_data['target']

        print(f"   훈련: {X_train.shape}, 테스트: {X_test.shape}")
        print(f"   피처 수: {len(feature_cols)}")

        # 5. 다중 시간프레임 앙상블 훈련
        print("\n🤖 5단계: 고급 앙상블 모델 훈련")
        ensemble = MultiTimeframeEnsemble()

        # 교차 검증으로 훈련
        cv_scores = ensemble.train_with_cross_validation(X_train, y_train, feature_cols)

        # 6. 예측 및 평가
        print("\n🔮 6단계: 예측 및 평가")
        predictions = ensemble.predict_ensemble(X_test)

        # 결과 평가
        results = {}
        for name, pred in predictions.items():
            r2 = r2_score(y_test, pred)
            rmse = np.sqrt(mean_squared_error(y_test, pred))

            # 방향성 정확도
            direction_acc = np.mean(np.sign(pred) == np.sign(y_test))

            # Sharpe-like ratio
            pred_returns = pred
            sharpe_like = np.mean(pred_returns) / (np.std(pred_returns) + 1e-10)

            results[name] = {
                'R2': r2,
                'RMSE': rmse,
                'Direction_Accuracy': direction_acc,
                'Sharpe_Like': sharpe_like
            }

        # 결과 출력
        results_df = pd.DataFrame(results).T
        results_df = results_df.sort_values('R2', ascending=False)

        print("\n📊 향상된 모델 성능:")
        print(results_df.round(4))

        # 최고 성능 모델
        best_model = results_df.index[0]
        best_r2 = results_df.loc[best_model, 'R2']
        best_direction = results_df.loc[best_model, 'Direction_Accuracy']

        print(f"\n🏆 최고 성능 모델: {best_model.upper()}")
        print(f"   - R² Score: {best_r2:.4f}")
        print(f"   - 방향성 정확도: {best_direction:.4f} ({best_direction*100:.1f}%)")
        print(f"   - Sharpe-like: {results_df.loc[best_model, 'Sharpe_Like']:.4f}")

        # 최신 예측
        latest_pred = predictions[best_model][-1]
        current_price = final_data['Close'].iloc[-2]  # 마지막에서 두번째 (타겟 때문에)
        predicted_price = current_price * (1 + latest_pred)

        print(f"\n🔮 최신 예측:")
        print(f"   - 예상 수익률: {latest_pred:.4f} ({latest_pred*100:.2f}%)")
        print(f"   - 현재 가격: {current_price:,.0f}원")
        print(f"   - 예상 가격: {predicted_price:,.0f}원")

        # 투자 신호
        if latest_pred > 0.015:
            signal = "🟢 강한 매수"
        elif latest_pred > 0.005:
            signal = "🔵 매수"
        elif latest_pred > -0.005:
            signal = "⚪ 중립"
        elif latest_pred > -0.015:
            signal = "🟡 매도"
        else:
            signal = "🔴 강한 매도"

        print(f"   - 투자 신호: {signal}")

        # 중요 피처 분석 (Random Forest 기준)
        if 'rf' in ensemble.models:
            feature_importance = ensemble.models['rf'].feature_importances_
            top_features = sorted(zip(feature_cols, feature_importance),
                                  key=lambda x: x[1], reverse=True)[:10]

            print(f"\n📈 주요 예측 인자 (상위 10개):")
            for i, (feature, importance) in enumerate(top_features, 1):
                print(f"   {i:2d}. {feature}: {importance:.4f}")

        print("\n" + "=" * 60)
        print("🎯 고성능 예측 시스템 V2.0 완료!")

        return {
            'data': final_data,
            'predictions': predictions,
            'results': results_df,
            'ensemble': ensemble,
            'feature_importance': top_features if 'rf' in ensemble.models else None
        }

    except Exception as e:
        print(f"\n❌ 시스템 오류: {e}")
        import traceback
        traceback.print_exc()
        return None


In [ ]:
def create_enhanced_visualization(results):
    """향상된 시각화 실행"""
    if not results:
        print("❌ 결과 데이터가 없습니다.")
        return

    print("🎨 고급 시각화 시작!")
    print("=" * 50)

    # 시각화 객체 생성
    visualizer = PredictionVisualizer(results)

    # 1. 종합 대시보드
    print("\n📊 1. 종합 분석 대시보드 생성...")
    dashboard = visualizer.plot_comprehensive_analysis(days_ahead=30)

    # 2. 3D 예측 표면
    print("\n🌐 2. 3D 예측 표면 생성...")
    surface_3d = visualizer.plot_3d_prediction_surface()

    # 3. 예측 타임라인
    print("\n📅 3. 예측 타임라인 생성...")
    timeline = visualizer.plot_prediction_timeline(days_ahead=30)

    # 4. 추가 분석 차트
    print("\n📈 4. 추가 분석 차트 생성...")

    # 모델 성능 비교
    if 'results' in results:
        fig_performance = go.Figure()

        models = list(results['results'].index)
        r2_scores = list(results['results']['R2'])
        direction_acc = list(results['results']['Direction_Accuracy'])

        fig_performance.add_trace(
            go.Bar(
                x=models,
                y=r2_scores,
                name='R² Score',
                marker_color='lightblue',
                yaxis='y'
            )
        )

        fig_performance.add_trace(
            go.Scatter(
                x=models,
                y=direction_acc,
                mode='lines+markers',
                name='방향성 정확도',
                line=dict(color='red', width=3),
                marker=dict(size=10),
                yaxis='y2'
            )
        )

        fig_performance.update_layout(
            title="🏆 모델 성능 비교",
            xaxis_title="모델",
            yaxis=dict(title="R² Score", side="left"),
            yaxis2=dict(title="방향성 정확도", side="right", overlaying="y"),
            template="plotly_white",
            height=400
        )

        fig_performance.show()

    # 5. 위험-수익 분석
    if 'predictions' in results:
        print("\n⚡ 5. 위험-수익 분석...")

        test_start_idx = int(len(results['data']) * 0.8)
        actual_returns = results['data']['Close'].pct_change().iloc[test_start_idx+1:]

        fig_risk = go.Figure()

        for model_name, predictions in results['predictions'].items():
            if len(predictions) == len(actual_returns):
                # 누적 수익률 계산
                cumulative_pred = (1 + pd.Series(predictions)).cumprod()
                cumulative_actual = (1 + actual_returns).cumprod()

                fig_risk.add_trace(
                    go.Scatter(
                        x=actual_returns.index,
                        y=cumulative_pred,
                        name=f'{model_name} 예측',
                        mode='lines'
                    )
                )

        # 실제 누적 수익률
        fig_risk.add_trace(
            go.Scatter(
                x=actual_returns.index,
                y=(1 + actual_returns).cumprod(),
                name='실제 수익률',
                line=dict(color='black', width=3),
                mode='lines'
            )
        )

        fig_risk.update_layout(
            title="📊 누적 수익률 비교",
            xaxis_title="날짜",
            yaxis_title="누적 수익률",
            template="plotly_white",
            height=400
        )

        fig_risk.show()

    print("\n🎨 모든 시각화 완료!")
    print("💾 대시보드가 HTML 파일로 저장되었습니다.")

    return {
        'dashboard': dashboard,
        'surface_3d': surface_3d,
        'timeline': timeline,
        'visualizer': visualizer
    }


In [ ]:
# 통합 시스템 실행
print("🚀 삼성전자 AI 예측 시스템 통합 실행!")
print("=" * 60)

# 1. 예측 시스템 실행
enhanced_results = run_enhanced_system()

# 2. 시각화 실행 (결과가 있는 경우)
if enhanced_results:
    print("\n" + "🎨" * 20)
    visualization_results = create_enhanced_visualization(enhanced_results)

    print(f"\n🎯 최종 완료!")
    print(f"📊 예측 시스템: {'✅ 성공' if enhanced_results else '❌ 실패'}")
    print(f"🎨 시각화: {'✅ 성공' if visualization_results else '❌ 실패'}")
    print(f"💾 HTML 대시보드: samsung_prediction_dashboard.html")

else:
    print("❌ 예측 시스템 실행 실패")